# Delta Lake — Basics
Leitura do CSV de transações, escrita como Delta table e operações fundamentais.

In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .master("local[*]")
    .appName("delta-basics")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} + Delta Lake pronto")

Spark 3.5.0 + Delta Lake pronto


## 1. Ler o CSV e criar a camada Bronze (Delta table)

In [2]:
DELTA_PATH = "/home/jovyan/work/delta-tables"
CSV_PATH   = "/home/jovyan/data/customer_transactions.csv"

raw = spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH)
raw.printSchema()
raw.show(5)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- price: string (nullable = true)
 |-- tax: string (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_city: string (nullable = true)

+--------------+-----------+----------------+----------+------------+--------+------+-----+-------------------+------------------+--------------------+---------------+----------------+-------------+
|transaction_id|customer_id|transaction_date|product_id|product_name|quantity| price|  tax|customer_first_name|customer_last_name|      customer_email| customer_phone|custome

In [3]:
# Escrever como Delta table (bronze)
(
    raw.write
    .format("delta")
    .mode("overwrite")
    .save(f"{DELTA_PATH}/bronze/customer_transactions")
)
print("Bronze escrita com sucesso")

Bronze escrita com sucesso


## 2. Ler a Delta table

In [4]:
bronze = spark.read.format("delta").load(f"{DELTA_PATH}/bronze/customer_transactions")
print(f"Registros: {bronze.count()}")
bronze.show(5)

Registros: 100
+--------------+-----------+----------------+----------+------------+--------+------+-----+-------------------+------------------+--------------------+---------------+----------------+-------------+
|transaction_id|customer_id|transaction_date|product_id|product_name|quantity| price|  tax|customer_first_name|customer_last_name|      customer_email| customer_phone|customer_country|customer_city|
+--------------+-----------+----------------+----------+------------+--------+------+-----+-------------------+------------------+--------------------+---------------+----------------+-------------+
|          1001|      501.0|      2023-07-11|       101|   Product A|     1.0| 76.27| 8.23|               John|               Doe|john.doe@example.com|+1-234-567-8901|             USA|     New York|
|          1002|      502.0|      2023-07-12|       102|   Product B|     3.0|119.16|17.06|               Jane|             Smith|jane.smith@exampl...|+1-234-567-8902|          Canada|     

## 3. Criar camada Silver (limpeza e tipagem)

In [5]:
silver = (
    bronze
    .dropDuplicates(["transaction_id"])
    .withColumn("customer_id",       F.col("customer_id").cast("integer"))
    .withColumn("transaction_date",  F.to_date("transaction_date"))
    .withColumn("quantity",          F.col("quantity").cast("integer"))
    .withColumn("total_amount",      F.round(F.col("price") + F.col("tax"), 2))
    .withColumn("full_name",         F.concat_ws(" ", "customer_first_name", "customer_last_name"))
    .drop("customer_first_name", "customer_last_name")
)

silver.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/silver/transaction")
print("Silver escrita com sucesso")
silver.show(5)

Silver escrita com sucesso
+--------------+-----------+----------------+----------+------------+--------+------+-----+--------------------+---------------+----------------+-------------+------------+--------------+
|transaction_id|customer_id|transaction_date|product_id|product_name|quantity| price|  tax|      customer_email| customer_phone|customer_country|customer_city|total_amount|     full_name|
+--------------+-----------+----------------+----------+------------+--------+------+-----+--------------------+---------------+----------------+-------------+------------+--------------+
|          1001|        501|      2023-07-11|       101|   Product A|       1| 76.27| 8.23|john.doe@example.com|+1-234-567-8901|             USA|     New York|        84.5|      John Doe|
|          1002|        502|      2023-07-12|       102|   Product B|       3|119.16|17.06|jane.smith@exampl...|+1-234-567-8902|          Canada|      Toronto|      136.22|    Jane Smith|
|          1003|        503|     

## 4. Inspecionar o _delta_log (transaction log)

In [6]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, f"{DELTA_PATH}/silver/transaction")
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------+
|version|timestamp              |operation|operationParameters                            |
+-------+-----------------------+---------+-----------------------------------------------+
|2      |2026-05-18 16:41:30.11 |WRITE    |{mode -> Overwrite, partitionBy -> []}         |
|1      |2026-05-18 16:23:39.949|UPDATE   |{predicate -> ["(customer_country#2758 = UK)"]}|
|0      |2026-05-18 16:22:45.028|WRITE    |{mode -> Overwrite, partitionBy -> []}         |
+-------+-----------------------+---------+-----------------------------------------------+



## 5. Atualizar dados (UPDATE via DeltaTable API)

In [7]:
# Normalizar país: UK → United Kingdom
dt.update(
    condition=F.col("customer_country") == "UK",
    set={"customer_country": F.lit("United Kingdom")}
)

spark.read.format("delta").load(f"{DELTA_PATH}/silver/transaction") \
    .filter(F.col("customer_country") == "United Kingdom") \
    .show(5)

+--------------+-----------+----------------+----------+------------+--------+-----------+-------+--------------------+---------------+----------------+-------------+------------+--------------+
|transaction_id|customer_id|transaction_date|product_id|product_name|quantity|      price|    tax|      customer_email| customer_phone|customer_country|customer_city|total_amount|     full_name|
+--------------+-----------+----------------+----------+------------+--------+-----------+-------+--------------------+---------------+----------------+-------------+------------+--------------+
|          1003|        503|      2023-07-13|       103|   Product C|       1|     287.25|  26.36|bob.brown@example...|+1-234-567-8903|  United Kingdom|       London|      313.61|     Bob Brown|
|          1066|        506|      2023-07-11|       101|   Product A|    NULL|     224.76|Fifteen|emma.bennett@emai...|+1-234-567-8966|  United Kingdom|   Manchester|        NULL|  Emma Bennett|
|          1072|        5